# Rectangle

An axis-aligned box in 2D, optionally with rounded corners: plates, electrodes and walls.

`Rectangle(midpoint, side_lengths, fillet=r)` rounds the four corners with radius `r`, up to
half the shorter side, where it becomes a stadium. Rounded corners keep the surface field of an
electrode bounded. With or without the fillet, the level set is the exact signed distance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from simgeometry import Rectangle

plates = {
    "sharp": Rectangle(midpoint=(0.0, 0.0), side_lengths=(2.0, 0.6)),
    "fillet 0.15": Rectangle(midpoint=(0.0, 0.0), side_lengths=(2.0, 0.6), fillet=0.15),
    "stadium": Rectangle(midpoint=(0.0, 0.0), side_lengths=(2.0, 0.6), fillet=0.3),
}
corner = (0.97, 0.27)
for name, plate in plates.items():
    print(f"{name:12s} near the corner inside: {plate.contains(corner)!s:5s}", end="  ")
    print(f"distance from (1.2, 0.5): {plate.signed_distance((1.2, 0.5)):.4f}")
assert np.isclose(plates["sharp"].signed_distance((1.2, 0.5)), np.hypot(0.2, 0.2))

## Outlines and offset contours

Each rectangle with its outline and the contours of its level set at 0, 0.15 and 0.3 outside:
the offset contours of an exact distance are rounded even around sharp corners.

In [ ]:
x = np.linspace(-1.6, 1.6, 321)
y = np.linspace(-0.9, 0.9, 181)
X, Y = np.meshgrid(x, y, indexing="ij")
points = np.stack((X.ravel(), Y.ravel()))

fig, axes = plt.subplots(3, 1, figsize=(6, 6.5), layout="constrained", sharex=True)
for ax, (name, plate) in zip(axes, plates.items(), strict=True):
    distance = plate.signed_distance(points).reshape(X.shape)
    ax.contour(X, Y, distance, levels=[0.15, 0.3], colors="0.6", linewidths=0.8)
    (loop,) = plate.outlines(max_spacing=0.02)
    ax.fill(*loop, color="tab:blue", alpha=0.4)
    ax.plot(*np.c_[loop, loop[:, :1]], color="tab:blue")
    ax.set(ylabel="y", title=name)
    ax.set_aspect("equal")
axes[-1].set_xlabel("x")
plt.show()